# MedVision-AI: Kaggle GPU Cloud Environment & Execution Pipeline

This notebook provides the Cloud GPU Host Environment for MedVision-AI.
Source Repository: https://github.com/SwastikPandey1024/MedVision-AI

MLOps Strategy: Local laptop executes fast CPU smoke tests (execution_mode: development). This Kaggle notebook executes Phase 1 EDA & Full Dataset Ingestion (execution_mode: full).

In [ ]:
# Cell 1: Environment & Dynamic Hardware Verification Script
import os
import sys
import platform

# Force Keras 3 TensorFlow Backend
os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import keras

print("=" * 75)
print("MedVision-AI Kaggle GPU Environment Report")
print("=" * 75)

# Environment versions
print(f"Python Version       : {platform.python_version()} ({sys.executable})")
print(f"TensorFlow Version   : {tf.__version__}")
print(f"Keras Version        : {keras.__version__}")
print(f"Keras Backend        : {os.environ.get('KERAS_BACKEND', 'tensorflow')}")

# Dynamic GPU Hardware Auto-Detection
gpus = tf.config.list_physical_devices('GPU')
gpu_count = len(gpus)
gpu_available = gpu_count > 0

print(f"\nGPU Availability     : {'YES (GPU Enabled)' if gpu_available else 'NO (CPU Fallback)'}")
print(f"GPU Device Count     : {gpu_count}")

if gpu_available:
    for i, gpu in enumerate(gpus):
        print(f"\n--- GPU Device #{i+1} Details ---")
        print(f"Device Identifier    : {gpu.name}")
        try:
            details = tf.config.experimental.get_device_details(gpu)
            hardware_name = details.get('device_name', 'NVIDIA GPU')
            print(f"Hardware Model       : {hardware_name}")
            compute_cap = details.get('compute_capability', None)
            if compute_cap:
                print(f"Compute Capability   : {compute_cap}")
        except Exception as err:
            print(f"Hardware Details     : {err}")
        
        # Detect Available VRAM Memory info
        try:
            gpu_mem = tf.config.experimental.get_memory_info(gpu.name)
            current_mb = gpu_mem.get('current', 0) / (1024 * 1024)
            peak_mb = gpu_mem.get('peak', 0) / (1024 * 1024)
            print(f"VRAM Memory Status   : Current: {current_mb:.2f} MB | Peak: {peak_mb:.2f} MB")
        except Exception:
            pass
else:
    print("\n[NOTE] No GPU device discovered in current Kaggle kernel session.")

print("=" * 75)

In [ ]:
# Cell 2: Repository Clone & Package Installation
!git clone https://github.com/SwastikPandey1024/MedVision-AI.git
%cd MedVision-AI
!pip install -e .

In [ ]:
# Cell 3: Phase 1 — RSNA Dataset Ingestion & EDA Report Execution
from medvision.data.dataset import find_dataset_root, parse_rsna_manifest
from medvision.data.validation import validate_manifest_integrity
from medvision.data.splits import create_patient_aware_splits, verify_zero_patient_leakage
from medvision.data.eda import generate_eda_report

print("=" * 75)
print("Phase 1: RSNA Dataset Ingestion & Data Engineering")
print("=" * 75)

# 1. Auto-detect RSNA Dataset Root Path
ds_root = find_dataset_root()
print(f"[1] Dataset Root Resolved: {ds_root}")

# 2. Parse RSNA Metadata Manifest
manifest_df = parse_rsna_manifest(ds_root)
print(f"[2] Manifest Parsed: {len(manifest_df)} unique patient records.")

# 3. Run Data Quality & Integrity Validation Engine
val_audit = validate_manifest_integrity(manifest_df)
print(f"[3] Data Quality Audit Valid: {val_audit['is_valid']} (Missing: {val_audit['missing_files_count']}, Dupes: {val_audit['duplicate_patients_count']})")

# 4. Generate EDA Report
eda_res = generate_eda_report(manifest_df)
print(f"[4] EDA Report Generated: Total={eda_res['total_unique_patients']}, Pneumonia={eda_res['class_distribution']['positive_pneumonia_count']}")

# 5. Create Patient-Aware Group Splits (70/15/15)
train_df, val_df, test_df = create_patient_aware_splits(manifest_df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)
leakage_audit = verify_zero_patient_leakage(train_df, val_df, test_df)
print(f"[5] Patient Leakage Audit: Zero Leakage = {leakage_audit['has_zero_leakage']}")
print(f"    Split Counts: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")

print("=" * 75)